# FloodShift — Standard Segmentation Baselines

**Notebook:** `04A_FloodShift_Standard_Baselines_8Models_READY`

This notebook trains and evaluates **8 standardized segmentation baselines**:

1. FCN-ResNet50
2. U-Net-ResNet34
3. U-Net++-ResNet34
4. DeepLabV3+-ResNet50
5. PSPNet-ResNet50
6. FPN-ResNet34
7. SegFormer-B0
8. SegFormer-B2

All models use the same geographic split, 12-channel multimodal input, 3-class target, 256×256 tile protocol, loss, optimizer, LR schedule, epoch budget, early stopping, and evaluation code.

**Kaggle inputs required:** attach both the saved output of `01_FloodShift_Data_Audit` and the final saved output of `03_FloodShift_Preprocessing`.

## Literature basis

The baseline set is grounded in both flood-mapping and semantic-segmentation literature.

- FCN-ResNet50 is the original deep-learning Sen1Floods11 baseline.
- U-Net is widely used for Sentinel-1 / Sentinel-2 flood mapping.
- DeepLabV3+ has been directly reproduced on Sen1Floods11.
- U-Net++ appears in published SAR water/flood benchmarking.
- SegFormer provides the modern transformer comparison, including recent flood-segmentation comparisons on Sen1Floods11.
- PSPNet and FPN provide standard multi-scale context controls used broadly in remote-sensing segmentation.

Representative references include Bonafilia et al. (Sen1Floods11, CVPRW 2020), Konapala et al. (S1/S2 flood mapping, 2021), Li et al. (Sen1Floods11 multimodal fusion, 2021), Ronneberger et al. (U-Net), Zhou et al. (UNet++), Chen et al. (DeepLabV3+), Zhao et al. (PSPNet), Lin et al. (FPN), and Xie et al. (SegFormer).

The notebook does **not** reuse reported scores from papers. Every model is retrained under the same FloodShift split and evaluation code.

In [ ]:
# ============================================================
# CELL: B-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================

import os
import sys
import gc
import json
import time
import random
import subprocess
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

import torchvision
from torchvision.models.segmentation import fcn_resnet50

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = torch.cuda.is_available()

print("=" * 100)
print("B-C01 — BASELINE ENVIRONMENT")
print("=" * 100)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("Device      :", DEVICE)

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print("GPU count   :", torch.cuda.device_count())

try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("\nInstalling segmentation-models-pytorch==0.5.0 ...")

    try:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "segmentation-models-pytorch==0.5.0",
            ]
        )
    except Exception as exc:
        raise RuntimeError(
            "Could not install segmentation-models-pytorch. "
            "Enable Internet for this Kaggle notebook and rerun B-C01."
        ) from exc

    importlib.invalidate_caches()
    import segmentation_models_pytorch as smp

print("SMP         :", getattr(smp, "__version__", "unknown"))

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_Baselines")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for path in [
    OUTPUT_ROOT,
    CHECKPOINT_ROOT,
    HISTORY_ROOT,
    METRIC_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: B-C02 — LOCATE FROZEN FLOODSHIFT ARTIFACTS
#
# Attach BOTH:
#   01_FloodShift_Data_Audit
#   03_FloodShift_Preprocessing FINAL saved output
# ============================================================

_file_cache = {}

def locate_by_saved_path(saved_path_or_name):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    matches = list(
        INPUT_ROOT.rglob(filename)
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not locate '{filename}' under /kaggle/input. "
            "Attach BOTH Notebook 01 and Notebook 03 outputs."
        )

    parent_hint = saved.parent.name.lower()

    hinted = [
        path
        for path in matches
        if parent_hint
        and parent_hint in str(path).lower()
    ]

    candidates = (
        hinted
        if hinted
        else matches
    )

    selected = sorted(
        candidates,
        key=lambda path: (
            len(str(path)),
            str(path)
        )
    )[0]

    _file_cache[cache_key] = selected

    return selected


FINAL_MANIFEST_PATH = locate_by_saved_path(
    "floodshift_preprocessing_manifest_v1.csv"
)

GLOBAL_LOCK_PATH = locate_by_saved_path(
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

OPTICAL_NORM_PATH = locate_by_saved_path(
    "optical_10band_train_normalization_exact.csv"
)

SAR_SCENE_STATS_PATH = locate_by_saved_path(
    "train_raw_sar_scene_statistics.csv"
)

print("=" * 100)
print("B-C02 — FROZEN ARTIFACTS FOUND")
print("=" * 100)

for path in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH,
]:
    print(" ", path)

with open(
    GLOBAL_LOCK_PATH,
    "r"
) as file:
    global_lock = json.load(file)

assert (
    global_lock["status"]
    ==
    "PERMANENTLY_FROZEN"
)

assert (
    global_lock["preprocessing_development_closed"]
    is True
)

print(
    "\nPreprocessing lock:",
    global_lock["status"]
)


In [ ]:
# ============================================================
# CELL: B-C03 — MANIFEST + FROZEN INPUT NORMALIZATION
# ============================================================

manifest = pd.read_csv(
    FINAL_MANIFEST_PATH
)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[
        manifest["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[
        manifest["split"] == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[
        manifest["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

# ------------------------------------------------------------
# Optical — frozen Notebook-03 normalization
# ------------------------------------------------------------

optical_norm = pd.read_csv(
    OPTICAL_NORM_PATH
)

OPTICAL_BANDS = optical_norm[
    "band_name"
].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",
]

assert (
    OPTICAL_BANDS
    ==
    EXPECTED_OPTICAL_BANDS
)

OPTICAL_MEAN = optical_norm[
    "mean"
].to_numpy(
    dtype=np.float32
)

OPTICAL_STD = optical_norm[
    "std"
].to_numpy(
    dtype=np.float32
)

# Source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12

OPTICAL_SOURCE_INDICES = np.array(
    [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        11,
        12,
    ],
    dtype=np.int64
)

# ------------------------------------------------------------
# SAR — pooled TRAIN statistics from Notebook-03 C05
# ------------------------------------------------------------

sar_stats = pd.read_csv(
    SAR_SCENE_STATS_PATH
)

assert len(sar_stats) == 314

valid_counts = sar_stats[
    "valid_pixels"
].to_numpy(
    dtype=np.float64
)

def pooled_mean_std(
    mean_column,
    std_column
):
    means = sar_stats[
        mean_column
    ].to_numpy(
        dtype=np.float64
    )

    stds = sar_stats[
        std_column
    ].to_numpy(
        dtype=np.float64
    )

    valid = (
        np.isfinite(means)
        &
        np.isfinite(stds)
        &
        (valid_counts > 0)
    )

    n = valid_counts[valid]
    mu = means[valid]
    sigma = stds[valid]

    total_n = n.sum()

    global_mean = (
        np.sum(
            n * mu
        )
        /
        total_n
    )

    second_moment = (
        np.sum(
            n
            *
            (
                sigma ** 2
                +
                mu ** 2
            )
        )
        /
        total_n
    )

    variance = max(
        second_moment
        -
        global_mean ** 2,
        0.0
    )

    return (
        float(global_mean),
        float(np.sqrt(variance))
    )


VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db"
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db"
)

SAR_MEAN = np.array(
    [
        VV_MEAN,
        VH_MEAN
    ],
    dtype=np.float32
)

SAR_STD = np.array(
    [
        VV_STD,
        VH_STD
    ],
    dtype=np.float32
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 100)
print("B-C03 — DATA + NORMALIZATION")
print("=" * 100)

print(
    "Train / Val / Test:",
    len(train_df),
    len(val_df),
    len(test_df)
)

print(
    "\nOptical bands:",
    OPTICAL_BANDS
)

print(
    "Optical mean:",
    OPTICAL_MEAN
)

print(
    "Optical std :",
    OPTICAL_STD
)

print(
    "\nSAR mean [VV,VH]:",
    SAR_MEAN
)

print(
    "SAR std  [VV,VH]:",
    SAR_STD
)

print(
    "\nBaseline input = 12 channels "
    "(2 SAR + 10 optical)"
)


## Three-class target

This notebook uses the revised FloodShift segmentation target:

- `0 = non-flood`
- `1 = permanent water`
- `2 = temporary flood`
- `255 = ignore`

For every valid Sen1Floods11 hand-label pixel:

- JRC permanent water `== 1` → class `1`
- otherwise flood hand label `== 1` → class `2`
- otherwise → class `0`
- hand-label `-1` → `255`

The proposed-model notebook must reuse the same constructor so the baseline comparison is valid.

In [ ]:
# ============================================================
# CELL: B-C04 — DATASET + THREE-CLASS TARGET
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar":
            locate_by_saved_path(
                row["sar_raw_path"]
            ),

        "optical":
            locate_by_saved_path(
                row["optical_raw_path"]
            ),

        "flood":
            locate_by_saved_path(
                row["flood_label_path"]
            ),

        "jrc":
            locate_by_saved_path(
                row["jrc_water_path"]
            ),
    }


def build_three_class_target(
    flood,
    jrc
):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8
    )

    valid = (
        flood != -1
    )

    # valid non-flood by default
    target[valid] = 0

    # permanent water
    target[
        valid
        &
        (jrc == 1)
    ] = 1

    # temporary flood
    target[
        valid
        &
        (jrc != 1)
        &
        (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False
    )

    valid = np.isfinite(
        sar
    ).all(
        axis=0
    )

    output = np.zeros_like(
        sar,
        dtype=np.float32
    )

    for channel in range(2):
        channel_valid = (
            valid
            &
            np.isfinite(
                sar[channel]
            )
        )

        output[
            channel,
            channel_valid
        ] = (
            sar[
                channel,
                channel_valid
            ]
            -
            SAR_MEAN[channel]
        ) / SAR_STD[channel]

    return output, valid


def normalize_optical(
    optical_13
):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False
    )

    valid = np.all(
        np.isfinite(
            optical_13
        )
        &
        (
            optical_13
            != 0
        ),
        axis=0
    )

    reflectance = (
        optical_13
        /
        10000.0
    )

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    output = (
        optical_10
        -
        OPTICAL_MEAN[
            :,
            None,
            None
        ]
    ) / OPTICAL_STD[
        :,
        None,
        None
    ]

    output = output.astype(
        np.float32,
        copy=False
    )

    output[
        :,
        ~valid
    ] = 0.0

    return output, valid


def augment_pair(
    image,
    target
):
    if np.random.rand() < 0.5:
        image = np.flip(
            image,
            axis=2
        )

        target = np.flip(
            target,
            axis=1
        )

    if np.random.rand() < 0.5:
        image = np.flip(
            image,
            axis=1
        )

        target = np.flip(
            target,
            axis=0
        )

    k = np.random.randint(
        0,
        4
    )

    if k:
        image = np.rot90(
            image,
            k=k,
            axes=(1, 2)
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1)
        )

    return (
        image.copy(),
        target.copy()
    )


class FloodShiftBaselineDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        train_mode=False
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            *
            TILES_PER_SCENE
        )

    def __getitem__(
        self,
        index
    ):
        scene_index = (
            index
            //
            TILES_PER_SCENE
        )

        tile_index = (
            index
            %
            TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(
                np.float32
            )

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(
                np.float32
            )

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window
            )

        sar_norm, _ = normalize_sar(
            sar
        )

        optical_norm, _ = normalize_optical(
            optical_13
        )

        image = np.concatenate(
            [
                sar_norm,
                optical_norm
            ],
            axis=0
        ).astype(
            np.float32
        )

        target = build_three_class_target(
            flood,
            jrc
        )

        if self.train_mode:
            image, target = augment_pair(
                image,
                target
            )

        return {
            "image":
                torch.from_numpy(
                    image
                ),

            "target":
                torch.from_numpy(
                    target.astype(
                        np.int64
                    )
                ),

            "scene_id":
                scene_id,

            "tile_index":
                tile_index,
        }


In [ ]:
# ============================================================
# CELL: B-C05 — TRAIN TARGET AUDIT + CLASS WEIGHTS
# ============================================================

class_counts = np.zeros(
    NUM_CLASSES,
    dtype=np.int64
)

ignored_count = 0

for position, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):
    paths = resolve_scene_files(
        row
    )

    with rasterio.open(
        paths["flood"]
    ) as src:
        flood = src.read(1)

    with rasterio.open(
        paths["jrc"]
    ) as src:
        jrc = src.read(1)

    target = build_three_class_target(
        flood,
        jrc
    )

    for class_id in range(
        NUM_CLASSES
    ):
        class_counts[
            class_id
        ] += int(
            (
                target
                ==
                class_id
            ).sum()
        )

    ignored_count += int(
        (
            target
            ==
            IGNORE_INDEX
        ).sum()
    )

    if (
        position == 1
        or
        position % 75 == 0
        or
        position == len(train_df)
    ):
        print(
            f"[{position:03d}/{len(train_df)}] "
            "target audit"
        )

valid_total = class_counts.sum()

frequencies = (
    class_counts
    /
    valid_total
)

class_weights_np = (
    1.0
    /
    np.maximum(
        frequencies,
        1e-12
    )
)

class_weights_np = (
    class_weights_np
    /
    class_weights_np.mean()
)

CLASS_WEIGHTS = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

target_audit = pd.DataFrame(
    {
        "class_id":
            np.arange(
                NUM_CLASSES
            ),

        "class_name":
            CLASS_NAMES,

        "pixel_count":
            class_counts,

        "frequency":
            frequencies,

        "loss_weight":
            class_weights_np,
    }
)

print(
    "\n"
    +
    "=" * 100
)

print(
    "B-C05 — TRAIN 3-CLASS DISTRIBUTION"
)

print(
    "=" * 100
)

print(
    target_audit.to_string(
        index=False
    )
)

print(
    "\nIgnored pixels:",
    f"{ignored_count:,}"
)

print(
    "Class weights:",
    CLASS_WEIGHTS.detach().cpu().numpy()
)

target_audit.to_csv(
    METRIC_ROOT
    /
    "train_target_class_distribution.csv",
    index=False
)

assert (
    class_counts
    >
    0
).all()


In [ ]:
# ============================================================
# CELL: B-C06 — DATALOADERS + ONE-BATCH CHECK
# ============================================================

BATCH_SIZE = 8
NUM_WORKERS = 2


def seed_worker(
    worker_id
):
    worker_seed = (
        torch.initial_seed()
        %
        (2 ** 32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


train_dataset = FloodShiftBaselineDataset(
    train_df,
    train_mode=True
)

val_dataset = FloodShiftBaselineDataset(
    val_df,
    train_mode=False
)

test_dataset = FloodShiftBaselineDataset(
    test_df,
    train_mode=False
)


def make_loaders(
    seed=SEED
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    common = {
        "num_workers":
            NUM_WORKERS,

        "pin_memory":
            torch.cuda.is_available(),

        "worker_init_fn":
            seed_worker,

        "persistent_workers":
            (
                NUM_WORKERS
                >
                0
            ),
    }

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=generator,
        **common
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


train_loader, val_loader, test_loader = make_loaders()

batch = next(
    iter(train_loader)
)

print("=" * 100)
print("B-C06 — DATA PIPELINE")
print("=" * 100)

print(
    "Train tiles:",
    len(train_dataset)
)

print(
    "Val tiles  :",
    len(val_dataset)
)

print(
    "Test tiles :",
    len(test_dataset)
)

print(
    "Image batch:",
    tuple(
        batch["image"].shape
    )
)

print(
    "Target:",
    tuple(
        batch["target"].shape
    )
)

print(
    "Target values:",
    torch.unique(
        batch["target"]
    )
)

print(
    "Finite:",
    torch.isfinite(
        batch["image"]
    ).all().item()
)

assert (
    batch["image"].shape[1:]
    ==
    (
        12,
        256,
        256
    )
)

assert (
    batch["target"].shape[1:]
    ==
    (
        256,
        256
    )
)

assert torch.isfinite(
    batch["image"]
).all()


In [ ]:
# ============================================================
# CELL: B-C07 — EIGHT STANDARD BASELINE MODELS
# ============================================================

class TorchvisionFCNWrapper(
    nn.Module
):
    def __init__(
        self,
        in_channels=12,
        num_classes=3
    ):
        super().__init__()

        model = fcn_resnet50(
            weights=None,
            weights_backbone=None,
            num_classes=num_classes,
            aux_loss=False
        )

        old_conv = model.backbone.conv1

        model.backbone.conv1 = nn.Conv2d(
            in_channels,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        nn.init.kaiming_normal_(
            model.backbone.conv1.weight,
            mode="fan_out",
            nonlinearity="relu"
        )

        self.model = model

    def forward(
        self,
        x
    ):
        return self.model(
            x
        )["out"]


def build_model(
    model_name
):
    if model_name == "FCN-ResNet50":
        model = TorchvisionFCNWrapper(
            in_channels=12,
            num_classes=NUM_CLASSES
        )

    elif model_name == "U-Net-ResNet34":
        model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "U-Net++-ResNet34":
        model = smp.UnetPlusPlus(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "DeepLabV3+-ResNet50":
        model = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "PSPNet-ResNet50":
        model = smp.PSPNet(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "FPN-ResNet34":
        model = smp.FPN(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "SegFormer-B0":
        model = smp.Segformer(
            encoder_name="mit_b0",
            encoder_weights=None,
            decoder_segmentation_channels=256,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    elif model_name == "SegFormer-B2":
        model = smp.Segformer(
            encoder_name="mit_b2",
            encoder_weights=None,
            decoder_segmentation_channels=768,
            in_channels=12,
            classes=NUM_CLASSES,
            activation=None
        )

    else:
        raise KeyError(
            model_name
        )

    return model


BASELINE_MODELS = [
    "FCN-ResNet50",
    "U-Net-ResNet34",
    "U-Net++-ResNet34",
    "DeepLabV3+-ResNet50",
    "PSPNet-ResNet50",
    "FPN-ResNet34",
    "SegFormer-B0",
    "SegFormer-B2",
]


print("=" * 100)
print("B-C07 — MODEL SANITY CHECK")
print("=" * 100)

dummy = torch.zeros(
    1,
    12,
    256,
    256,
    dtype=torch.float32
)

summary_rows = []

for model_name in BASELINE_MODELS:
    model = build_model(
        model_name
    ).eval()

    with torch.no_grad():
        logits = model(
            dummy
        )

    if (
        logits.shape[-2:]
        !=
        (
            256,
            256
        )
    ):
        logits = F.interpolate(
            logits,
            size=(
                256,
                256
            ),
            mode="bilinear",
            align_corners=False
        )

    assert (
        logits.shape
        ==
        (
            1,
            NUM_CLASSES,
            256,
            256
        )
    )

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    summary_rows.append(
        {
            "model":
                model_name,

            "trainable_parameters":
                parameter_count,
        }
    )

    print(
        f"{model_name:24s} -> "
        f"{tuple(logits.shape)} | "
        f"{parameter_count / 1e6:.2f} M params"
    )

    del model
    gc.collect()

model_summary = pd.DataFrame(
    summary_rows
)

model_summary.to_csv(
    METRIC_ROOT
    /
    "baseline_model_parameter_counts.csv",
    index=False
)


In [ ]:
# ============================================================
# CELL: B-C08 — COMMON LOSS + METRICS
# ============================================================

CE_WEIGHT = 0.5
DICE_WEIGHT = 0.5

cross_entropy = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX
)


def multiclass_dice_loss(
    logits,
    target,
    eps=1e-6
):
    probabilities = torch.softmax(
        logits,
        dim=1
    )

    valid = (
        target
        !=
        IGNORE_INDEX
    )

    safe_target = target.clone()
    safe_target[
        ~valid
    ] = 0

    one_hot = F.one_hot(
        safe_target,
        num_classes=NUM_CLASSES
    ).permute(
        0,
        3,
        1,
        2
    ).float()

    valid_float = valid.unsqueeze(
        1
    ).float()

    probabilities = (
        probabilities
        *
        valid_float
    )

    one_hot = (
        one_hot
        *
        valid_float
    )

    intersection = (
        probabilities
        *
        one_hot
    ).sum(
        dim=(
            0,
            2,
            3
        )
    )

    denominator = (
        probabilities.sum(
            dim=(
                0,
                2,
                3
            )
        )
        +
        one_hot.sum(
            dim=(
                0,
                2,
                3
            )
        )
    )

    dice = (
        2.0
        *
        intersection
        +
        eps
    ) / (
        denominator
        +
        eps
    )

    return (
        1.0
        -
        dice.mean()
    )


def common_loss(
    logits,
    target
):
    ce = cross_entropy(
        logits,
        target
    )

    dice = multiclass_dice_loss(
        logits,
        target
    )

    total = (
        CE_WEIGHT
        *
        ce
        +
        DICE_WEIGHT
        *
        dice
    )

    return (
        total,
        ce.detach(),
        dice.detach()
    )


def confusion_from_batch(
    logits,
    target
):
    prediction = torch.argmax(
        logits,
        dim=1
    )

    valid = (
        target
        !=
        IGNORE_INDEX
    )

    prediction = prediction[
        valid
    ].detach().cpu()

    target_valid = target[
        valid
    ].detach().cpu()

    encoded = (
        target_valid
        *
        NUM_CLASSES
        +
        prediction
    )

    confusion = torch.bincount(
        encoded,
        minlength=(
            NUM_CLASSES
            *
            NUM_CLASSES
        )
    ).reshape(
        NUM_CLASSES,
        NUM_CLASSES
    )

    return confusion.numpy().astype(
        np.int64
    )


def metrics_from_confusion(
    confusion
):
    confusion = confusion.astype(
        np.float64
    )

    true_positive = np.diag(
        confusion
    )

    ground_truth = confusion.sum(
        axis=1
    )

    predicted = confusion.sum(
        axis=0
    )

    false_positive = (
        predicted
        -
        true_positive
    )

    false_negative = (
        ground_truth
        -
        true_positive
    )

    iou = (
        true_positive
        /
        np.maximum(
            true_positive
            +
            false_positive
            +
            false_negative,
            1.0
        )
    )

    precision = (
        true_positive
        /
        np.maximum(
            true_positive
            +
            false_positive,
            1.0
        )
    )

    recall = (
        true_positive
        /
        np.maximum(
            true_positive
            +
            false_negative,
            1.0
        )
    )

    dice = (
        2.0
        *
        true_positive
        /
        np.maximum(
            2.0
            *
            true_positive
            +
            false_positive
            +
            false_negative,
            1.0
        )
    )

    accuracy = (
        true_positive.sum()
        /
        np.maximum(
            confusion.sum(),
            1.0
        )
    )

    result = {
        "mIoU":
            float(
                np.mean(iou)
            ),

        "macro_dice":
            float(
                np.mean(dice)
            ),

        "macro_precision":
            float(
                np.mean(precision)
            ),

        "macro_recall":
            float(
                np.mean(recall)
            ),

        "accuracy":
            float(
                accuracy
            ),
    }

    for class_id, class_name in enumerate(
        CLASS_NAMES
    ):
        result[
            f"iou_{class_name}"
        ] = float(
            iou[class_id]
        )

        result[
            f"dice_{class_name}"
        ] = float(
            dice[class_id]
        )

        result[
            f"precision_{class_name}"
        ] = float(
            precision[class_id]
        )

        result[
            f"recall_{class_name}"
        ] = float(
            recall[class_id]
        )

    return result


In [ ]:
# ============================================================
# CELL: B-C09 — COMMON TRAINING / EVALUATION FUNCTIONS
# ============================================================

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6

BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0


def run_epoch(
    model,
    loader,
    optimizer=None,
    scaler=None
):
    is_train = (
        optimizer
        is not None
    )

    if is_train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    running_ce = 0.0
    running_dice = 0.0
    batch_count = 0

    confusion = np.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES
        ),
        dtype=np.int64
    )

    for batch in loader:
        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            DEVICE,
            non_blocking=True
        )

        if is_train:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            is_train
        ):
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):
                logits = model(
                    images
                )

                if (
                    logits.shape[-2:]
                    !=
                    targets.shape[-2:]
                ):
                    logits = F.interpolate(
                        logits,
                        size=targets.shape[-2:],
                        mode="bilinear",
                        align_corners=False
                    )

                loss, ce, dice_loss = common_loss(
                    logits,
                    targets
                )

            if is_train:
                scaler.scale(
                    loss
                ).backward()

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

        running_loss += float(
            loss.detach().cpu()
        )

        running_ce += float(
            ce.cpu()
        )

        running_dice += float(
            dice_loss.cpu()
        )

        batch_count += 1

        confusion += confusion_from_batch(
            logits,
            targets
        )

    metrics = metrics_from_confusion(
        confusion
    )

    metrics.update(
        {
            "loss":
                running_loss
                /
                max(
                    batch_count,
                    1
                ),

            "ce_loss":
                running_ce
                /
                max(
                    batch_count,
                    1
                ),

            "dice_loss":
                running_dice
                /
                max(
                    batch_count,
                    1
                ),
        }
    )

    return metrics


@torch.no_grad()
def evaluate_model(
    model,
    loader
):
    return run_epoch(
        model=model,
        loader=loader,
        optimizer=None,
        scaler=None
    )


def train_one_baseline(
    model_name
):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            SEED
        )

    train_loader, val_loader, test_loader = make_loaders(
        seed=SEED
    )

    model = build_model(
        model_name
    ).to(
        DEVICE
    )

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP_ENABLED
    )

    safe_name = (
        model_name
        .replace(
            "+",
            "plus"
        )
        .replace(
            "/",
            "_"
        )
        .replace(
            " ",
            "_"
        )
    )

    checkpoint_path = (
        CHECKPOINT_ROOT
        /
        f"{safe_name}_best.pt"
    )

    history_path = (
        HISTORY_ROOT
        /
        f"{safe_name}_history.csv"
    )

    best_val_miou = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    training_start = time.perf_counter()

    print(
        "\n"
        +
        "=" * 110
    )

    print(
        "TRAINING:",
        model_name
    )

    print(
        "=" * 110
    )

    print(
        "Parameters:",
        f"{parameter_count / 1e6:.2f} M"
    )

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):
        epoch_start = time.perf_counter()

        train_metrics = run_epoch(
            model,
            train_loader,
            optimizer=optimizer,
            scaler=scaler
        )

        val_metrics = evaluate_model(
            model,
            val_loader
        )

        scheduler.step(
            val_metrics["mIoU"]
        )

        current_lr = optimizer.param_groups[
            0
        ]["lr"]

        epoch_record = {
            "epoch":
                epoch,

            "lr":
                current_lr,

            **{
                f"train_{key}":
                    value

                for key, value
                in train_metrics.items()
            },

            **{
                f"val_{key}":
                    value

                for key, value
                in val_metrics.items()
            },

            "epoch_sec":
                (
                    time.perf_counter()
                    -
                    epoch_start
                ),
        }

        history.append(
            epoch_record
        )

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train loss {train_metrics['loss']:.4f} | "
            f"val mIoU {val_metrics['mIoU']:.4f} | "
            f"temp IoU {val_metrics['iou_temporary_flood']:.4f} | "
            f"perm IoU {val_metrics['iou_permanent_water']:.4f} | "
            f"lr {current_lr:.2e}"
        )

        if (
            val_metrics["mIoU"]
            >
            best_val_miou
            +
            1e-6
        ):
            best_val_miou = val_metrics[
                "mIoU"
            ]

            best_epoch = epoch

            epochs_without_improvement = 0

            torch.save(
                {
                    "model_name":
                        model_name,

                    "epoch":
                        epoch,

                    "parameter_count":
                        parameter_count,

                    "state_dict":
                        model.state_dict(),

                    "val_metrics":
                        val_metrics,
                },
                checkpoint_path
            )

        else:
            epochs_without_improvement += 1

        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False
        )

        if (
            epochs_without_improvement
            >=
            EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping at epoch {epoch}; "
                f"best epoch = {best_epoch}"
            )

            break

    training_sec = (
        time.perf_counter()
        -
        training_start
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint[
            "state_dict"
        ]
    )

    best_val_metrics = evaluate_model(
        model,
        val_loader
    )

    test_metrics = evaluate_model(
        model,
        test_loader
    )

    result = {
        "model":
            model_name,

        "best_epoch":
            best_epoch,

        "parameters":
            parameter_count,

        "training_minutes":
            training_sec
            /
            60.0,

        **{
            f"val_{key}":
                value

            for key, value
            in best_val_metrics.items()
        },

        **{
            f"test_{key}":
                value

            for key, value
            in test_metrics.items()
        },

        "checkpoint":
            str(
                checkpoint_path
            ),
    }

    print(
        "\nBEST VALIDATION mIoU:",
        f"{best_val_metrics['mIoU']:.4f}"
    )

    print(
        "TEST mIoU:",
        f"{test_metrics['mIoU']:.4f}"
    )

    print(
        "TEST temporary IoU:",
        f"{test_metrics['iou_temporary_flood']:.4f}"
    )

    print(
        "TEST permanent IoU:",
        f"{test_metrics['iou_permanent_water']:.4f}"
    )

    del model
    del train_loader
    del val_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


## Common training protocol

- Input: 12-channel early fusion = raw SAR VV/VH + frozen 10-band optical
- Reliability channel: **not used**
- Tile size: 256×256
- Train / validation / test tiles: 1256 / 272 / 232
- Pretraining: none
- Batch size: 8
- Optimizer: AdamW
- Initial LR: `3e-4`
- Weight decay: `1e-4`
- Loss: `0.5 × weighted Cross-Entropy + 0.5 × macro Dice`
- Maximum epochs: 30
- Early stopping patience: 6
- Model selection: best validation mIoU
- Test evaluation: once after best validation checkpoint

The baseline architecture table is intentionally run on **Raw SAR**. FloodShift should therefore be compared to this table using its own **raw-SAR configuration**. SAR filtering/despeckling belongs in a separate preprocessing ablation table.

In [ ]:
# ============================================================
# CELL: B-C10 — RUN ALL EIGHT BASELINES
#
# Long-running cell.
# Results are checkpointed after every model.
# ============================================================

RESULTS_PATH = (
    METRIC_ROOT
    /
    "baseline_8model_results.csv"
)

completed_results = []

if RESULTS_PATH.exists():
    existing_results = pd.read_csv(
        RESULTS_PATH
    )

    completed_results = existing_results.to_dict(
        "records"
    )

    completed_names = set(
        existing_results[
            "model"
        ].tolist()
    )

    print(
        "Resuming existing run. Completed:",
        completed_names
    )

else:
    completed_names = set()


for model_name in BASELINE_MODELS:
    if model_name in completed_names:
        print(
            "\nSKIP — already completed:",
            model_name
        )

        continue

    try:
        result = train_one_baseline(
            model_name
        )

        completed_results.append(
            result
        )

        pd.DataFrame(
            completed_results
        ).to_csv(
            RESULTS_PATH,
            index=False
        )

    except RuntimeError as exc:
        if (
            "out of memory"
            in str(exc).lower()
        ):
            print(
                f"\nCUDA OOM while training {model_name}."
            )

            print(
                "Batch size is intentionally not auto-changed "
                "because the benchmark protocol must remain uniform."
            )

        raise


print(
    "\n"
    +
    "=" * 110
)

print(
    "ALL EIGHT BASELINES COMPLETED ✅"
)

print(
    "=" * 110
)

print(
    "Saved:",
    RESULTS_PATH
)


In [ ]:
# ============================================================
# CELL: B-C11 — FINAL BASELINE TABLE
# ============================================================

baseline_results = pd.read_csv(
    RESULTS_PATH
)

assert (
    set(
        baseline_results[
            "model"
        ]
    )
    ==
    set(
        BASELINE_MODELS
    )
)

report_columns = [
    "model",
    "parameters",
    "best_epoch",
    "training_minutes",
    "val_mIoU",
    "test_mIoU",
    "test_iou_non_flood",
    "test_iou_permanent_water",
    "test_iou_temporary_flood",
    "test_macro_dice",
    "test_macro_precision",
    "test_macro_recall",
    "test_accuracy",
]

baseline_report = (
    baseline_results[
        report_columns
    ]
    .sort_values(
        "test_mIoU",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print("=" * 120)
print("FLOODSHIFT — STANDARD 8-MODEL BASELINE RESULTS")
print("=" * 120)

print(
    baseline_report.to_string(
        index=False
    )
)

FINAL_REPORT_PATH = (
    METRIC_ROOT
    /
    "baseline_8model_final_comparison.csv"
)

baseline_report.to_csv(
    FINAL_REPORT_PATH,
    index=False
)

best_model = baseline_report.iloc[
    0
]

print(
    "\nBest standard baseline by TEST mIoU:"
)

print(
    "  Model:",
    best_model[
        "model"
    ]
)

print(
    "  mIoU:",
    f"{best_model['test_mIoU']:.4f}"
)

print(
    "  Temporary-flood IoU:",
    f"{best_model['test_iou_temporary_flood']:.4f}"
)

print(
    "  Permanent-water IoU:",
    f"{best_model['test_iou_permanent_water']:.4f}"
)

print(
    "\nSaved:",
    FINAL_REPORT_PATH
)


In [ ]:
# ============================================================
# CELL: B-C12 — SAVE PAPER / REPRODUCIBILITY PROTOCOL
# ============================================================

protocol = {
    "experiment":
        "FloodShift standard segmentation baselines",

    "dataset":
        "Sen1Floods11 hand-labelled subset",

    "retained_scenes":
        440,

    "geographic_split": {
        "train":
            314,

        "validation":
            68,

        "test":
            58,
    },

    "models":
        BASELINE_MODELS,

    "input": {
        "channels":
            12,

        "sar":
            [
                "VV",
                "VH"
            ],

        "sar_variant":
            "Raw SAR",

        "optical":
            OPTICAL_BANDS,

        "optical_reliability_channel":
            False,

        "fusion":
            "early channel concatenation",
    },

    "target": {
        "classes": {
            "0":
                "non_flood",

            "1":
                "permanent_water",

            "2":
                "temporary_flood",

            "255":
                "ignore",
        },

        "permanent_water_rule":
            "valid hand-label AND JRC == 1",

        "temporary_flood_rule":
            "valid hand-label AND JRC != 1 AND flood == 1",
    },

    "training": {
        "tile_size":
            TILE_SIZE,

        "tiles_per_scene":
            TILES_PER_SCENE,

        "batch_size":
            BATCH_SIZE,

        "optimizer":
            "AdamW",

        "learning_rate":
            BASE_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "max_epochs":
            MAX_EPOCHS,

        "early_stopping_patience":
            EARLY_STOPPING_PATIENCE,

        "scheduler":
            "ReduceLROnPlateau(validation mIoU)",

        "loss":
            "0.5 weighted CrossEntropy + 0.5 macro Dice",

        "pretrained_weights":
            False,

        "amp":
            AMP_ENABLED,

        "seed":
            SEED,
    },

    "model_selection":
        "best validation mIoU",

    "test_policy":
        "evaluate test only after validation-based checkpoint selection",

    "metrics": [
        "mIoU",
        "class-wise IoU",
        "macro Dice",
        "class-wise Dice",
        "macro precision",
        "macro recall",
        "accuracy",
    ],

    "preprocessing_lock":
        "Notebook 03 permanent global lock",
}

PROTOCOL_PATH = (
    OUTPUT_ROOT
    /
    "baseline_experiment_protocol.json"
)

with open(
    PROTOCOL_PATH,
    "w"
) as file:
    json.dump(
        protocol,
        file,
        indent=2
    )

print("=" * 100)
print("BASELINE NOTEBOOK COMPLETE ✅")
print("=" * 100)

print(
    "Results :",
    FINAL_REPORT_PATH
)

print(
    "Protocol:",
    PROTOCOL_PATH
)

print(
    "Models  :",
    CHECKPOINT_ROOT
)

print(
    "\nSave this Kaggle notebook version WITH outputs."
)


# Paper-use note

Use `baseline_8model_final_comparison.csv` as the conventional baseline table.

Recommended columns:

- Model
- Parameters
- mIoU
- IoU — Non-flood
- IoU — Permanent water
- IoU — Temporary flood
- Macro Dice/F1
- Precision
- Recall
- Accuracy

Keep the following in the **ablation** table rather than calling them conventional baselines:

- fixed 0.5 / 0.5 SAR-optical fusion
- router-only
- transformer-only
- no-reliability version
- skip-attention ablation
- channel-attention ablation
- SAR preprocessing variants
- loss-function experiments
- differential-learning-rate experiments
- ensemble experiments